In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-proj-123456"

In [33]:
import json
import textwrap
from copy import deepcopy
from typing import Any, Dict, List, Optional
from openai import OpenAI

In [34]:
# =========================
# Cell 2: client + config
# =========================

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = os.environ.get("OPENAI_MODEL", "gpt-5")
MAX_ITERS = 12
DEBUG = True

In [35]:
# =========================
# Cell 3: skill definition
# =========================

SKILLS = {
    "analyze_python_file": {
        "name": "analyze_python_file",
        "description": "Use this skill when the user asks for Python code review or complexity/risk analysis.",
        "steps": [
            "Inspect the target file and identify relevant functions/classes.",
            "Summarize the structure of the code.",
            "Analyze complexity, correctness risks, and readability issues.",
            "Produce the final answer in the required review format."
        ],
        "completion_criteria": [
            "The file has been inspected.",
            "Main functions/classes have been identified.",
            "Complexity and risk notes are included.",
            "A user-facing final answer is ready."
        ]
    },
    "summarize_markdown_file": {
        "name": "summarize_markdown_file",
        "description": "Use this skill when the user asks for a Markdown document summary, outline, or section-by-section recap.",
        "steps": [
            "Open the target Markdown file and inspect its headings and sections.",
            "Identify the document's main topics and structure.",
            "Summarize the key points in concise prose.",
            "Produce a user-facing summary or outline."
        ],
        "completion_criteria": [
            "The Markdown file has been inspected.",
            "The section structure has been identified.",
            "The main ideas have been summarized.",
            "A user-facing Markdown summary is ready."
        ]
    },
    "inspect_directory_structure": {
        "name": "inspect_directory_structure",
        "description": "Use this skill when the user asks for a folder inventory, project layout overview, or file organization summary.",
        "steps": [
            "List the contents of the requested directory.",
            "Group important files and subdirectories by purpose.",
            "Identify notable patterns, missing pieces, or unusual structure.",
            "Produce a user-facing directory overview."
        ],
        "completion_criteria": [
            "The target directory has been listed.",
            "Important folders and files have been categorized.",
            "Notable structure observations have been captured.",
            "A user-facing directory overview is ready."
        ]
    }
}

In [36]:
# =========================
# Cell 4: helper functions
# =========================

def pretty(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, indent=2)

def safe_json_loads(s: str) -> Dict[str, Any]:
    try:
        return json.loads(s)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model did not return valid JSON:\n{s}") from e

def trim_text(text: str, max_chars: int = 8000) -> str:
    if text is None:
        return ""
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...[truncated]..."

In [37]:
# =========================
# Cell 5: state init
# =========================

def initialize_state(user_query: str) -> Dict[str, Any]:
    return {
        "user_query": user_query,
        "goal": user_query,
        "done": False,
        "final_answer": None,

        "active_skill": None,
        "skill_steps": [], 
        "current_step_idx": None,

        "facts": [],
        "progress_notes": [],
        "next_action_hint": None,

        "history": [],              # action + raw_obs + interpreted_obs
        "raw_observations": [],
        "errors": [],
    }

In [38]:
# =========================
# Cell 6: local tools
# =========================

def tool_list_dir(path: str = ".") -> Dict[str, Any]:
    try:
        entries = sorted(os.listdir(path))
        return {"ok": True, "path": path, "entries": entries}
    except Exception as e:
        return {"ok": False, "error": str(e), "path": path}

def tool_read_file(path: str) -> Dict[str, Any]:
    try:
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        return {
            "ok": True,
            "path": path,
            "content": content,
        }
    except Exception as e:
        return {"ok": False, "error": str(e), "path": path}

def tool_write_file(path: str, content: str) -> Dict[str, Any]:
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        return {"ok": True, "path": path, "n_chars": len(content)}
    except Exception as e:
        return {"ok": False, "error": str(e), "path": path}

TOOLS = {
    "list_dir": tool_list_dir,
    "read_file": tool_read_file,
    "write_file": tool_write_file,
}

In [39]:
# =========================
# Cell 7: LLM JSON caller
# =========================

def call_llm_json(system_prompt: str, user_prompt: str) -> Dict[str, Any]:
    """
    Ask the model to return JSON only.
    """
    response = client.responses.create(
        model=MODEL, 
        instructions=system_prompt, 
        input=user_prompt, 
    )
    text = response.output_text.strip()
    return safe_json_loads(text)

In [40]:
# =========================
# Cell 8: prompt builders
# =========================

PLANNER_SYSTEM_PROMPT = """
    You are the planner for a small autonomous coding agent.

    You must decide exactly ONE next action.
    Do not execute tools yourself.
    Do not write explanations outside JSON.

    Allowed action types:
    - load_skill
    - tool
    - reason
    - respond

    Rules:
    1. A skill step is a goal, not an atomic action.
    2. Use "tool" when you need new external information.
    3. Use "reason" when you already have enough raw information and need to transform it into understanding.
    4. Step completion is judged by the interpreter/state updater, not by you.
    5. Use "respond" only when the whole user task is complete.
    6. Return valid JSON only.

    JSON schema:
    {
      "thought": "short reasoning",
      "next_action": {
        "type": "load_skill | tool | reason | respond",
        "name": "optional name",
        "args": {},
        "why": "short reason"
      }

      For type \"respond\", args must include:
      {
        "response": "final user-facing answer"
      }
    }
    """.strip()


def build_planner_prompt(state: Dict[str, Any]) -> str:
    active_skill = state["active_skill"]
    current_step = None
    if active_skill is not None and state["current_step_idx"] is not None:
        idx = state["current_step_idx"]
        if 0 <= idx < len(state["skill_steps"]):
            current_step = state["skill_steps"][idx]

    recent_history = state["history"][-2:]
    
    prompt = f"""
        User request:
        {state["user_query"]}

        Current state:
        - goal: {state["goal"]}
        - active_skill: {active_skill}
        - current_step_idx: {state["current_step_idx"]}
        - current_step: {current_step}
        - known facts: {pretty(state["facts"][-8:])}
        - next_action_hint: {state["next_action_hint"]}
        - recent history: {pretty(recent_history)}

        Available tools:
        - list_dir(path=".")
        - read_file(path)
        - write_file(path, content)

        Available skills:
        {pretty(SKILLS)}

        Decide the single best next action.
        Do not decide whether a step is complete; that will be inferred after execution.
        When choosing respond, put the final user-facing answer in next_action.args.response.
        Return JSON only.
        """.strip()
    return prompt

In [41]:
# =========================
# Cell 9: interpreter prompt
# =========================

INTERPRETER_SYSTEM_PROMPT = """
    You are the interpreter for a small autonomous coding agent.

    Your job is to convert a raw observation into structured task-relevant meaning.
    You are also responsible for judging whether the CURRENT STEP is complete.

    Return JSON only with this schema:
    {
    "facts": ["..."],
    "step_complete": true or false,
    "progress_note": "short note",
    "next_action_hint": "short hint"
    }

    Facts should be short, atomic, and reusable.
    Prefer facts that help evaluate the current step.
    Do not restate the full raw observation.
    If the raw observation already contains high-quality structured facts, reuse them instead of rewriting them.
    If nothing useful was learned, return an empty facts list.
    """.strip()


def build_interpreter_prompt(
    state: Dict[str, Any],
    action: Dict[str, Any],
    raw_observation: Dict[str, Any],
) -> str:
    active_skill = state["active_skill"]
    current_step = None
    if active_skill is not None and state["current_step_idx"] is not None:
        idx = state["current_step_idx"]
        if 0 <= idx < len(state["skill_steps"]):
            current_step = state["skill_steps"][idx]

    prompt = f"""
        Goal:
        {state["goal"]}

        Active skill:
        {active_skill}

        Current step:
        {current_step}

        Action just executed:
        {pretty(action)}

        Raw observation:
        {pretty(raw_observation)}

        Interpret the observation for the CURRENT STEP only.
        Extract only short, reusable facts from the observation.
        If the observation already includes structured facts, preserve them unless they need small cleanup.
        Decide whether the current step is now complete based on the action, observation, and current step goal.
        Return JSON only.
        """.strip()
    return prompt

In [42]:
# =========================
# Cell 10: planner
# =========================

def planner(state: Dict[str, Any]) -> Dict[str, Any]:
    prompt = build_planner_prompt(state)
    decision = call_llm_json(
        system_prompt=PLANNER_SYSTEM_PROMPT,
        user_prompt=prompt,
    )
    return decision

In [43]:
# =========================
# Cell 11: executor
# =========================

def executor(action: Dict[str, Any], state: Dict[str, Any]) -> Dict[str, Any]:
    a_type = action["type"]
    name = action.get("name")
    args = action.get("args", {})

    if a_type == "load_skill":
        skill = SKILLS[name]
        return {
            "type": "skill_loaded",
            "skill_name": name,
            "skill": deepcopy(skill),
        }

    if a_type == "tool":
        if name not in TOOLS:
            return {"type": "error", "error": f"Unknown tool: {name}"}
        result = TOOLS[name](**args)
        return {
            "type": "tool_result",
            "tool_name": name,
            "tool_args": args,
            "result": result,
        }

    if a_type == "reason":
        # reasoning action = ask model to transform current evidence into useful beliefs
        reasoning_prompt = f"""
            You are a reasoning module for a coding agent.

            Current goal:
            {state["goal"]}

            Active skill:
            {state["active_skill"]}

            Current step:
            {state["skill_steps"][state["current_step_idx"]] if state["active_skill"] else None}

            Known facts:
            {pretty(state["facts"][-12:])}

            Recent raw observations:
            {pretty(state["raw_observations"][-2:])}

            Requested reasoning action:
            {pretty(action)}

            Return JSON only with:
            {{
            "reasoning_summary": "short summary",
            "facts": ["..."]
            }}
            """.strip()

        result = call_llm_json(
            system_prompt="You are a precise reasoning engine. Return JSON only.",
            user_prompt=reasoning_prompt,
        )
        return {
            "type": "reason_result",
            "reason_name": name,
            "result": result,
        }

    if a_type == "respond":
        response_text = args.get("response") or args.get("content", "")
        return {
            "type": "final_response",
            "response": response_text
        }

    return {"type": "error", "error": f"Unknown action type: {a_type}"}

In [44]:
# =========================
# Cell 12: interpreter
# =========================

def interpreter(
    state: Dict[str, Any],
    action: Dict[str, Any],
    raw_observation: Dict[str, Any],
) -> Dict[str, Any]:
    obs_type = raw_observation.get("type")

    if obs_type == "skill_loaded":
        steps = raw_observation["skill"]["steps"]
        return {
            "facts": [f"Loaded skill {raw_observation['skill_name']}."],
            "step_complete": False,
            "progress_note": f"Skill loaded with {len(steps)} steps.",
            "next_action_hint": f"Work on step 0: {steps[0]}",
        }

    if obs_type == "final_response":
        return {
            "facts": ["Final response prepared."],
            "step_complete": False,
            "progress_note": "Task complete.",
            "next_action_hint": None,
        }

    if obs_type == "error":
        return {
            "facts": [],
            "step_complete": False,
            "progress_note": f"Error: {raw_observation.get('error')}",
            "next_action_hint": "Recover from the error.",
        }

    if obs_type == "reason_result":
        reason_result = raw_observation.get("result", {})
        prompt = build_interpreter_prompt(state, action, raw_observation)
        judged = call_llm_json(
            system_prompt=INTERPRETER_SYSTEM_PROMPT,
            user_prompt=prompt,
        )
        return {
            "facts": reason_result.get("facts", []),
            "step_complete": judged.get("step_complete", False),
            "progress_note": judged.get("progress_note") or reason_result.get("reasoning_summary", "Reasoning completed."),
            "next_action_hint": judged.get("next_action_hint"),
        }

    prompt = build_interpreter_prompt(state, action, raw_observation)
    result = call_llm_json(
        system_prompt=INTERPRETER_SYSTEM_PROMPT,
        user_prompt=prompt,
    )
    return result

In [45]:
# =========================
# Cell 13: state updater
# =========================

def update_state(
    state: Dict[str, Any],
    decision: Dict[str, Any],
    raw_observation: Dict[str, Any],
    interpreted_observation: Dict[str, Any],
) -> Dict[str, Any]:
    
    state["raw_observations"].append(raw_observation)
    state["history"].append({
        "decision": decision,
        "raw_observation": raw_observation,
        "interpreted_observation": interpreted_observation,
    })

    for fact in interpreted_observation.get("facts", []):
        if fact not in state["facts"]:
            state["facts"].append(fact)

    note = interpreted_observation.get("progress_note")
    if note:
        state["progress_notes"].append(note)

    state["next_action_hint"] = interpreted_observation.get("next_action_hint")

    obs_type = raw_observation.get("type")
    if obs_type == "skill_loaded":
        skill = raw_observation["skill"]
        state["active_skill"] = raw_observation["skill_name"]
        state["skill_steps"] = skill["steps"]
        state["current_step_idx"] = 0

    elif interpreted_observation.get("step_complete"):
        if state["current_step_idx"] is not None:
            completed_step_idx = state["current_step_idx"]
            completed_step = None
            if 0 <= completed_step_idx < len(state["skill_steps"]):
                completed_step = state["skill_steps"][completed_step_idx]

            if completed_step is not None:
                completion_fact = f"Completed skill step {completed_step_idx}: {completed_step}"
                if completion_fact not in state["facts"]:
                    state["facts"].append(completion_fact)

            state["current_step_idx"] += 1
            if state["current_step_idx"] >= len(state["skill_steps"]):
                state["next_action_hint"] = "All skill steps finished; prepare final response."
            else:
                next_step = state["skill_steps"][state["current_step_idx"]]
                state["next_action_hint"] = f"Work on step {state['current_step_idx']}: {next_step}"

    elif obs_type == "final_response":
        state["final_answer"] = raw_observation["response"]
        state["done"] = True

    elif obs_type == "error":
        state["errors"].append(raw_observation.get("error", "unknown error"))

    return state

In [46]:
# =========================
# Cell 14: main agent loop
# =========================

def run_agent(user_query: str, max_iters: int = MAX_ITERS, debug: bool = DEBUG) -> Dict[str, Any]:
    state = initialize_state(user_query)

    for i in range(max_iters):
        if state["done"]:
            break
        decision = planner(state) 
        action = decision["next_action"]

        raw_observation = executor(action, state) 
        interpreted_observation = interpreter(state, action, raw_observation)

        state = update_state(
            state=state,
            decision=decision,
            raw_observation=raw_observation,
            interpreted_observation=interpreted_observation,
        )

        if debug:
            print("=" * 80)
            print(f"ITER {i+1}")
            print("- action")
            print(pretty(action))
            print("- raw_observation")
            print(trim_text(pretty(raw_observation), 2000))
            print("- interpreted_observation")
            print(pretty(interpreted_observation))
            print("- state summary")
            print(pretty({
                "active_skill": state["active_skill"],
                "current_step_idx": state["current_step_idx"],
                "next_action_hint": state["next_action_hint"],
                "n_facts": len(state["facts"]),
                "done": state["done"],
            }))

        if (
            state["active_skill"] is not None
            and state["current_step_idx"] is not None
            and state["current_step_idx"] >= len(state["skill_steps"])
            and not state["done"]
        ):
            pass

    if not state["done"]:
        state["final_answer"] = "Agent stopped before explicit completion."
    return state

In [47]:
# =========================
# Cell 16: run the agent
# =========================

query = "Please review example.py and tell me the complexity and the main risks."

state = run_agent(query, max_iters=10, debug=True)
print("\nFINAL ANSWER:\n")
print(state["final_answer"])

ITER 1
- action
{
  "type": "load_skill",
  "name": "analyze_python_file",
  "args": {},
  "why": "This skill provides the steps to inspect example.py, analyze complexity and risks, and produce the final review."
}
- raw_observation
{
  "type": "skill_loaded",
  "skill_name": "analyze_python_file",
  "skill": {
    "name": "analyze_python_file",
    "description": "Use this skill when the user asks for Python code review or complexity/risk analysis.",
    "steps": [
      "Inspect the target file and identify relevant functions/classes.",
      "Summarize the structure of the code.",
      "Analyze complexity, correctness risks, and readability issues.",
      "Produce the final answer in the required review format."
    ],
    "completion_criteria": [
      "The file has been inspected.",
      "Main functions/classes have been identified.",
      "Complexity and risk notes are included.",
      "A user-facing final answer is ready."
    ]
  }
}
- interpreted_observation
{
  "facts": 